In [ ]:
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from model import *

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with h5py.File("dataset/Dataset_Specific_Unlabelled.h5", "r") as f:
    for key in f.keys():
        print(key, f[key].shape)



PyTorch : 2.12.0.dev20260217+cu128
CUDA    : True
GPU     : NVIDIA GeForce RTX 5060 Laptop GPU
jet (60000, 125, 125, 8)


In [2]:
with h5py.File("dataset/Dataset_Specific_Unlabelled.h5", "r") as f:
    for key in f.keys():
        print(key, f[key].shape, f[key].dtype)

jet (60000, 125, 125, 8) float32


In [3]:
with h5py.File("dataset/Dataset_Specific_Labelled.h5", "r") as f:
    print("X shape:", f["jet"].shape)
    print("Y shape:", f["Y"].shape)
    print("Y dtype:", f["Y"].dtype)
    y_sample = f["Y"][:100].flatten().tolist() 
    print("Y unique values:", set(y_sample))
    print("Y min:", min(y_sample))
    print("Y max:", max(y_sample))

X shape: (10000, 125, 125, 8)
Y shape: (10000, 1)
Y dtype: float32
Y unique values: {0.0, 1.0}
Y min: 0.0
Y max: 1.0


In [4]:
class H5Dataset(Dataset):
    def __init__(self, file_path, x_key="jet", threshold=0.0):
        self.file      = h5py.File(file_path, "r")
        self.X         = self.file[x_key]
        self.threshold = threshold

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x    = torch.tensor(self.X[idx], dtype=torch.float32)  # (H, W, C)
        x    = x.permute(2, 0, 1)                              # (C, H, W)
        mask = (x.abs().sum(dim=0, keepdim=True) > self.threshold)  # (1, H, W)
        return x, mask



In [5]:
class DenseAutoencoder(nn.Module):
    def __init__(self, in_channels=8):
        super().__init__()

        # Encoder — same structure, no masking
        self.enc1  = nn.Sequential(nn.Conv2d(in_channels, 32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU())
        self.enc2  = nn.Sequential(nn.Conv2d(32,          64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU())
        self.down1 = nn.Sequential(nn.Conv2d(64,          64,  3, padding=1, stride=2), nn.BatchNorm2d(64),  nn.ReLU())
        self.enc3  = nn.Sequential(nn.Conv2d(64,          128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.down2 = nn.Sequential(nn.Conv2d(128,         128, 3, padding=1, stride=2), nn.BatchNorm2d(128), nn.ReLU())

        # Decoder — same as yours
        self.up1  = nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1)
        self.dec1 = nn.Sequential(nn.Conv2d(64,  64,          3, padding=1), nn.BatchNorm2d(64),  nn.ReLU())
        self.up2  = nn.ConvTranspose2d(64,  32, 3, stride=2, padding=1, output_padding=1)
        self.dec2 = nn.Sequential(nn.Conv2d(32,  in_channels, 3, padding=1))

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):  # ← no mask needed!
        # Encoder
        z = self.enc1(x)
        z = self.enc2(z)
        z = self.down1(z)
        z = self.enc3(z)
        z = self.down2(z)

        latent = z

        # Decoder
        z = self.relu(self.up1(z))
        z = self.dec1(z)
        z = self.relu(self.up2(z))
        x_hat = self.dec2(z)

        x_hat = x_hat[:, :, :x.shape[2], :x.shape[3]]

        return x_hat, latent

In [6]:
# same training loop but no mask
def dense_loss(x, x_hat, latent, l1_lambda=1e-2):
    recon_loss = F.mse_loss(x_hat, x)  # ← no masking, full reconstruction
    sparsity   = torch.mean(torch.abs(latent))
    return recon_loss + l1_lambda * sparsity, recon_loss, sparsity

dense_model = DenseAutoencoder(in_channels=8).to(device)
optimizer   = optim.SGD(dense_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)
scheduler   = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

In [7]:
dataset   = H5Dataset("dataset/Dataset_Specific_Unlabelled.h5", x_key="jet")
loader    = DataLoader(dataset, batch_size=100, shuffle=True,
                       num_workers=0, pin_memory=True)

In [8]:
for epoch in range(100):
    dense_model.train()
    total_loss  = 0.0
    total_recon = 0.0
    total_sparse = 0.0

    for X, mask in loader:
        X, mask = X.to(device), mask.to(device)

        optimizer.zero_grad()
        X_hat, latent =dense_model(X)

        loss, recon, sparsity = dense_loss(X, X_hat, latent, l1_lambda=1e-2)
        loss.backward()
        optimizer.step()

        total_loss   += loss.item()
        total_recon  += recon.item()
        total_sparse += sparsity.item()

    scheduler.step()
    n = len(loader)
    print(f"Epoch {epoch+1:02d} | "
          f"Loss {total_loss/n:.4f} | "
          f"Recon {total_recon/n:.4f} | "
          f"Sparsity {total_sparse/n:.4f} | "
          f"LR {scheduler.get_last_lr()[0]:.2e}")

Epoch 01 | Loss 57.8464 | Recon 57.8445 | Sparsity 0.1846 | LR 9.50e-04
Epoch 02 | Loss 37.3771 | Recon 37.3744 | Sparsity 0.2642 | LR 9.02e-04
Epoch 03 | Loss 16.4283 | Recon 16.4251 | Sparsity 0.3193 | LR 8.57e-04
Epoch 04 | Loss 6.6143 | Recon 6.6109 | Sparsity 0.3414 | LR 8.15e-04
Epoch 05 | Loss 5.7771 | Recon 5.7736 | Sparsity 0.3507 | LR 7.74e-04
Epoch 06 | Loss 5.4140 | Recon 5.4104 | Sparsity 0.3560 | LR 7.35e-04
Epoch 07 | Loss 5.1114 | Recon 5.1078 | Sparsity 0.3588 | LR 6.98e-04
Epoch 08 | Loss 4.9132 | Recon 4.9096 | Sparsity 0.3609 | LR 6.63e-04
Epoch 09 | Loss 4.7623 | Recon 4.7586 | Sparsity 0.3629 | LR 6.30e-04
Epoch 10 | Loss 4.6529 | Recon 4.6493 | Sparsity 0.3646 | LR 5.99e-04
Epoch 11 | Loss 4.5718 | Recon 4.5682 | Sparsity 0.3659 | LR 5.69e-04
Epoch 12 | Loss 4.4804 | Recon 4.4767 | Sparsity 0.3667 | LR 5.40e-04
Epoch 13 | Loss 4.3938 | Recon 4.3901 | Sparsity 0.3669 | LR 5.13e-04
Epoch 14 | Loss 4.2928 | Recon 4.2891 | Sparsity 0.3677 | LR 4.88e-04
Epoch 15 | Los

In [ ]:
torch.save({
        'model_state_dict': dense_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
    }, "models/dense.pth")
print("\nClassifier saved → dense.pth")



Classifier saved → sparse_classifier.pth
